## Understand what the dataset needs to contain.
***
* `tickit_id` : Unique ID -- INC-001 format
* `created_at` : Timestamp when ticket was raised.
* `resolved_at` : Timestamp when ticket was closed.
* `priority` : P1 Critical/ P2 High / P3 Medium / P4 Low
* `category` : Network/ Hardware/ Software / Access/ Email
* `agent_id` : Which agent handle the ticket
* `resolution_hours` : Actual time taken to resolve
* `sla_breach` : Boolean -- did it breach the SLA threshold?

### SLA threshold - the business rules you are building around.
---
SLA (Service Level Agreement) defines the maximum time allowed to resolve a ticket by priority. This is the contract between IT and the business. Every analysis in this project traces back to these numbers.
* `P1 Critical`: 4 Hours (server down, network outage)
* `P2 High`: 8 Hours (app crash, security issues)
* `P3 Medium`: 24 Hours (slow performance, bug)
* `P4 Low`: 72 Hours (password reset, request)

In [4]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

In [5]:
# Dataset config
N_TICKETS = 5000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 6, 30)

AGENTS = [f"AGT-{str(i).zfill(5)}" for i in range(1, 16)] # 15 agents

PRIORITIES = ['P1-Critical', "P2-High","P3-Medium","P4-Low"]
PRIORITIES_WEIGHTS = [0.05, 0.20, 0.50, 0.25]

CATEGORIES = ['Network', 'Hardware', 'Software', 'Access Management', 'Email/Comm']
CATEGORY_WEIGHTS = [0.20, 0.15, 0.35, 0.20, 0.10]

# SLA thresholds in hours
SLA_THRESHOLDS = {
    "P1-Critical": 4,
    "P2-High":     8,
    "P3-Medium":   24,
    "P4-Low":      72,
}

RESOLUTION_MEANS = {
    "P1-Critical": 3.0,
    "P2-High":     6.5,
    "P3-Medium":   18.0,
    "P4-Low":      50.0,
}

RESOLUTION_STD = {
    "P1-Critical": 2.0,
    "P2-High":     3.5,
    "P3-Medium":   8.0,
    "P4-Low":      20.0,
}

In [6]:
# Generate Tickets
def random_business_datetime(start, end):
    """Generate realistic ticket creation time -- more ticket during business hours"""
    delta = end - start
    random_days = random.randint(0, delta.days)
    dt = start + timedelta(days=random_days)

    # Weight towards business hours (9am - 6pm) but allow 24/7 tickets.
    hour_weights = [0.5, 0.3, 0.2, 0.2, 0.2, 0.3, 0.8, 1.5,
                    2.5, 3.0, 3.0, 2.8, 2.5, 2.8, 2.5, 2.2,
                    2.0, 1.8, 1.5, 1.2, 1.0, 0.8, 0.7, 0.6]
    hour = random.choices(range(24), weights=hour_weights)[0]
    minute = random.randint(0, 59)
    return dt.replace(hour=hour, minute=minute, second=0)

rows = []
for i in range(N_TICKETS):
    ticket_id = f"INC-{str(i+1).zfill(5)}"
    priority = random.choices(PRIORITIES, weights=PRIORITIES_WEIGHTS)[0]
    category = random.choices(CATEGORIES, weights=CATEGORY_WEIGHTS)[0]

    # Assign agent - some agents handle more tickets (realistic workload imbalance).
    agent_weights = [3,3,3,2,2,2,2,2,1,1,1,1,1,1,1]
    agent = random.choices(AGENTS, weights=agent_weights)[0]

    created_at = random_business_datetime(START_DATE, END_DATE)

    # Resolution time - normally distributed around mean, median 0.5 h
    mean_hrs = RESOLUTION_MEANS[priority]
    str_hrs = RESOLUTION_STD[priority]
    resolution_hours = max(0.5, np.random.normal(mean_hrs, str_hrs))

    resolved_at = created_at + timedelta(hours=resolution_hours)

    # SLA breach
    sla_limit = SLA_THRESHOLDS[priority]
    sla_breach = resolution_hours > sla_limit

    # Reopened tickets (5% rate - add realism)
    reopened = random.random() < 0.05

    # Customer satisfaction (1-5, lower for breached tickets)
    if sla_breach:
        csat = random.choices([1, 2, 3, 4, 5], weights=[25, 35, 25, 10, 5])[0]
    else:
        csat = random.choices([1, 2, 3, 4, 5], weights=[5, 10, 20, 35, 30])[0]

    rows.append({
        'ticket_id': ticket_id,
        'created_at': created_at,
        'resolved_at': resolved_at,
        'priority': priority,
        'category': category,
        'agent_id': agent,
        'resolution_hours': round(resolution_hours, 2),
        'sla_limit_hours': sla_limit,
        'sla_breach': sla_breach,
        'reopened': reopened,
        'csat_score': csat,
    })

df = pd.DataFrame(rows)
df.to_csv("D:/programming/PROJECTS/da-projects/it-saas-consulting/it_servicedesk/data/tickets.csv", index=False)
print(f"Dataset generated: {df.shape[0]} tickets, {df.shape[1]} columns")

Dataset generated: 5000 tickets, 11 columns
